# Yale EMMLC Admission-Risk Logistic Regression

This notebook builds a Logistic Regression model to predict admission vs discharge using the cleaned Yale triage-time dataset. It compares train and test accuracy, recall, and precision in a single metrics dataframe.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("yale_clean_triage.csv")
TARGET = "disposition_admit"
RANDOM_STATE = 42

warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn.utils.extmath")

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

print(f"Dataset shape: {df.shape}")
print(f"Number of predictors: {X.shape[1]}")

Dataset shape: (560484, 220)
Number of predictors: 219


In [3]:
cc_columns = [column for column in X.columns if column.startswith("cc_")]
numeric_columns = [
    column
    for column in X.select_dtypes(include=[np.number]).columns
    if column not in cc_columns
]
categorical_columns = list(X.select_dtypes(include=["object", "category"]).columns)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (448387, 219)
Test shape: (112097, 219)


In [4]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

cc_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
        ("cc", cc_pipeline, cc_columns),
    ]
)

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

In [ ]:
_ = pipeline.fit(X_train, y_train)

In [6]:
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

metrics_df = pd.DataFrame([
    {
        "split": "train",
        "accuracy": accuracy_score(y_train, y_train_pred),
        "recall": recall_score(y_train, y_train_pred, zero_division=0),
        "precision": precision_score(y_train, y_train_pred, zero_division=0),
    },
    {
        "split": "test",
        "accuracy": accuracy_score(y_test, y_test_pred),
        "recall": recall_score(y_test, y_test_pred, zero_division=0),
        "precision": precision_score(y_test, y_test_pred, zero_division=0),
    },
])

metrics_df

,split,accuracy,recall,precision
0,train,0.787188,0.803931,0.607358
1,test,0.787211,0.801398,0.607810


Train and test accuracy are almost the same, so there is no major train/test performance gap.

Recall is high on both train and test, so the model catches many admitted patients.

Precision is moderate on both train and test, so some predicted admissions are actually discharged patients.